In [16]:
# =========================
# CELL 1 — Setup
# =========================

import math, time, random, copy, json, contextlib, os, zipfile, warnings
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from dataclasses import dataclass
from typing import Optional, Dict, List
from collections import defaultdict

warnings.filterwarnings("ignore")

# ── Deterministic seed ────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

# ── Device & AMP ──────────────────────────────────────────────────────────────
DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IS_CUDA = DEVICE.type == "cuda"
AMP     = IS_CUDA  # AMP only on CUDA — avoids fp16 NaN on CPU

_tv = tuple(int(x) for x in torch.__version__.split(".")[:2] if x.isdigit())
if _tv >= (2, 0):
    def _amp_ctx():     return torch.amp.autocast(device_type="cuda", enabled=AMP)
    def _make_scaler(): return torch.amp.GradScaler("cuda", init_scale=128.0) if AMP else None
else:
    def _amp_ctx():     return torch.cuda.amp.autocast(enabled=AMP)
    def _make_scaler(): return torch.cuda.amp.GradScaler(init_scale=128.0) if AMP else None

def _fp32_ctx():
    if IS_CUDA:
        return torch.amp.autocast(device_type="cuda", enabled=False)
    return contextlib.nullcontext()

print(f"✓ Device : {DEVICE}")
print(f"✓ AMP    : {AMP}  (GradScaler init_scale=128, not 65536)")
print(f"✓ PyTorch: {torch.__version__}")
print(f"✓ CUDA   : {torch.version.cuda if IS_CUDA else 'N/A'}")
print(f"✓ Seed   : {SEED}")
if IS_CUDA:
    print(f"✓ GPU    : {torch.cuda.get_device_name(0)}")

✓ Device : cuda
✓ AMP    : True  (GradScaler init_scale=128, not 65536)
✓ PyTorch: 2.10.0+cu128
✓ CUDA   : 12.8
✓ Seed   : 42
✓ GPU    : Tesla T4


In [17]:
# =========================
# CELL 2 — Config (Upgraded Capacity)
# =========================

from dataclasses import dataclass

@dataclass
class Cfg:
    # ── Model (High Capacity) ────────────────────────────────────────────────
    vocab_size:     int   = 64
    max_seq_len:    int   = 600
    d_model:        int   = 128     # INCREASED (was 64)
    n_heads:        int   = 4
    n_layers:       int   = 4       # INCREASED (was 1)
    ff_dim:         int   = 256     # INCREASED (was 128)
    dropout:        float = 0.1
    # Memory slots — Expanded to prevent collisions
    working_slots:  int   = 32      # INCREASED (was 8)
    episodic_slots: int   = 16      # INCREASED (was 4)
    semantic_slots: int   = 8       # INCREASED (was 4)
    n_clusters:     int   = 8       # INCREASED (was 4)
    n_summaries:    int   = 4       # INCREASED (was 2)
    cluster_temp:   float = 1.0
    causal:         bool  = True
    # ── Data ──────────────────────────────────────────────────────────────────
    n_train:        int   = 8000
    n_val:          int   = 1600
    n_test:         int   = 1600
    # ── HARD RUNTIME CONSTRAINTS ──────────────────────────────────────────────
    batch_size:     int   = 32      # INCREASED (was 16)
    epochs:         int   = 10      # INCREASED (was 5)
    steps_per_epoch:int   = 100
    train_seq_len:  int   = 128
    # ── Optimiser ─────────────────────────────────────────────────────────────
    lr:             float = 3e-4
    warmup_steps:   int   = 300     # INCREASED (was 100)
    weight_decay:   float = 1e-2
    grad_clip:      float = 1.0
    num_workers:    int   = 0

CFG = Cfg()

# ── Hard assertions ───────────────────────────────────────────────────────────
assert CFG.epochs          == 10,  "epochs must be 10"
assert CFG.d_model         == 128, "d_model must be 128"
assert CFG.n_layers        == 4,   "n_layers must be 4"
assert CFG.working_slots   == 32,  "working_slots must be 32"
assert CFG.d_model % CFG.n_heads == 0, "d_model must be divisible by n_heads"

print("✓ Config assertions passed (Upgraded Capacity Active)")
print(f"  batch={CFG.batch_size}, epochs={CFG.epochs}, "
      f"steps/epoch={CFG.steps_per_epoch}, train_seq={CFG.train_seq_len}")
print(f"  d_model={CFG.d_model}, n_layers={CFG.n_layers}, "
      f"working_slots={CFG.working_slots}, warmup_steps={CFG.warmup_steps}")

✓ Config assertions passed (Upgraded Capacity Active)
  batch=32, epochs=10, steps/epoch=100, train_seq=128
  d_model=128, n_layers=4, working_slots=32, warmup_steps=300


In [18]:
# =========================
# CELL 3 — Dataset
# =========================

class SCPDDataset(Dataset):
    """
    Selective Copy-Paste with Distraction — generated once, deterministically.

    Input  : [START(2)] m0..m3 [noise×nl] [QUERY(3)] m0..m3
    Target : PAD(0) everywhere EXCEPT last N_MARKERS positions = marker tokens.

    Why this tests memory:
      nl=10  → seq~20 : both models should learn this
      nl=40  → seq~50 : baseline attention starts diluting; HMCT working-mem helps
    """
    N_MARKERS = 4

    def __init__(self, n, nl, vocab_size, seed=0):
        rng   = random.Random(seed)
        vocab = list(range(4, vocab_size))   # tokens 0-3 are special
        self.seqs, self.nl = [], nl
        for _ in range(n):
            m   = rng.choices(vocab, k=self.N_MARKERS)
            z   = rng.choices(vocab, k=nl)
            seq = [2] + m + z + [3] + m
            tgt = [0] * (1 + self.N_MARKERS + nl + 1) + m
            self.seqs.append((seq, tgt))

    def __len__(self): return len(self.seqs)

    def __getitem__(self, i):
        x, y = self.seqs[i]
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)

def pad_collate(batch):
    xs, ys = zip(*batch)
    L  = max(t.shape[0] for t in xs)
    Xp = torch.stack([F.pad(t, (0, L - t.shape[0])) for t in xs])
    Yp = torch.stack([F.pad(t, (0, L - t.shape[0])) for t in ys])
    return Xp, Yp

def make_loaders(nl, cfg):
    kw = dict(batch_size=cfg.batch_size, collate_fn=pad_collate,
              num_workers=cfg.num_workers, pin_memory=IS_CUDA)
    tr = DataLoader(SCPDDataset(cfg.n_train, nl, cfg.vocab_size, SEED),
                    shuffle=True,  **kw)
    vl = DataLoader(SCPDDataset(cfg.n_val,   nl, cfg.vocab_size, SEED + 1),
                    shuffle=False, **kw)
    te = DataLoader(SCPDDataset(cfg.n_test,  nl, cfg.vocab_size, SEED + 2),
                    shuffle=False, **kw)
    return tr, vl, te

def scpd_acc(logits, targets, n_markers=SCPDDataset.N_MARKERS):
    with torch.no_grad():
        pred = logits.argmax(-1)[:, -n_markers:]
        tgt  = targets[:, -n_markers:]
        return (pred == tgt).all(1).float().mean().item()

# ── Dataset smoke checks ──────────────────────────────────────────────────────
_nl_smoke = 10
_ds_smoke = SCPDDataset(8, _nl_smoke, CFG.vocab_size, seed=0)
_xs, _ys  = _ds_smoke[0]
_expected_len = 1 + SCPDDataset.N_MARKERS + _nl_smoke + 1 + SCPDDataset.N_MARKERS
assert _xs.shape[0] == _expected_len, f"seq len error: {_xs.shape[0]} != {_expected_len}"
assert not torch.isnan(_xs.float()).any(), "NaN in dataset input"
assert not torch.isnan(_ys.float()).any(), "NaN in dataset target"
print(f"✓ Dataset validated: seq_len={_xs.shape[0]}, vocab={CFG.vocab_size}")
print(f"  n_train={CFG.n_train}, n_val={CFG.n_val}, n_test={CFG.n_test}")
del _ds_smoke, _xs, _ys

✓ Dataset validated: seq_len=20, vocab=64
  n_train=8000, n_val=1600, n_test=1600


In [19]:
# =========================
# CELL 4 — HMCT v8 Model (Stable BPTT)
# =========================

# ── Per-instance Diagnostics ──────────────────────────────────────────────────
class Diag:
    def __init__(self): self.d = defaultdict(list)

    def log(self, k, v):
        fv = float(v)
        if math.isfinite(fv): self.d[k].append(fv)

    def mean(self, k):
        v = self.d.get(k, [])
        return sum(v) / len(v) if v else 0.

    def reset(self): self.d.clear()

    def summary(self):
        ks = ["write_gate", "forget_gate", "compress_gate",
              "attn_entropy", "work_util", "epis_util"]
        return "  [diag] " + " | ".join(
            f"{k}={self.mean(k):.3f}" for k in ks if k in self.d)

# ── RBF kernel ────────────────────────────────────────────────────────────────
def _rbf_kernel(x: torch.Tensor) -> torch.Tensor:
    x32 = x.float()
    return torch.exp(
        x32 - x32.pow(2).sum(-1, keepdim=True) / (2.0 * x32.shape[-1])
    ).to(x.dtype)

# ── Linear Attention — Hybrid Detached BPTT ──────────────────────────────────
class LinAttn(nn.Module):
    def __init__(self, d, h, drop=0.1, causal=False):
        super().__init__()
        assert d % h == 0
        self.h, self.dk, self.causal = h, d // h, causal
        self.q_proj = nn.Linear(d, d, bias=False)
        self.k_proj = nn.Linear(d, d, bias=False)
        self.v_proj = nn.Linear(d, d, bias=False)
        self.out    = nn.Linear(d, d, bias=False)
        self.ln     = nn.LayerNorm(d)
        self.drop   = nn.Dropout(drop)
        for p in [self.q_proj, self.k_proj, self.v_proj, self.out]:
            nn.init.xavier_uniform_(p.weight)

    def _split(self, t, L):
        return t.view(t.shape[0], L, self.h, self.dk).transpose(1, 2)

    def _causal_recurrence(self, Q, K, V):
        """
        BPTT STRATEGY: Hybrid Detached Accumulator.
        We detach the long history (S_acc) to prevent noisy gradients and
        OOM over long sequences, but we keep the current step (S_step)
        IN the graph. This ensures k_proj and v_proj get clean, immediate
        gradient signals at every step t.
        """
        B, H, T, dk = Q.shape
        with _fp32_ctx():
            Qk  = _rbf_kernel(Q.float())
            Kk  = _rbf_kernel(K.float())
            V32 = V.float()

            S_acc = torch.zeros(B, H, dk, dk, device=Q.device, dtype=torch.float32)
            z_acc = torch.zeros(B, H, dk,     device=Q.device, dtype=torch.float32)

            out_steps = []
            for t in range(T):
                kt = Kk[:, :, t]
                vt = V32[:, :, t]
                qt = Qk[:, :, t]

                # Current step stays in graph
                S_step = torch.einsum("bhk,bhv->bhkv", kt, vt)
                z_step = kt

                # Combine detached history + in-graph current step
                S_use = S_acc.detach() + S_step
                z_use = z_acc.detach() + z_step

                num = torch.einsum("bhk,bhkv->bhv", qt, S_use)
                den = (qt * z_use).sum(-1, keepdim=True).clamp(min=1e-4)
                out_steps.append(num / den)

                # Detach for the next iteration
                S_acc = S_use.detach()
                z_acc = z_use.detach()

            out = torch.stack(out_steps, dim=2)
        return out.to(Q.dtype)

    def _noncausal(self, Q, K, V):
        with _fp32_ctx():
            Qk  = _rbf_kernel(Q.float())
            Kk  = _rbf_kernel(K.float())
            V32 = V.float()
            KV    = torch.einsum("bhmk,bhmv->bhkv", Kk, V32)
            z     = Kk.sum(2)
            num   = torch.einsum("bhtk,bhkv->bhtv", Qk, KV)
            denom = torch.einsum("bhtk,bhk->bht", Qk, z).unsqueeze(-1).clamp(1e-4)
        return (num / denom).to(Q.dtype)

    def forward(self, x, mem=None):
        B, T, D = x.shape
        src = mem if mem is not None else x
        Q   = self._split(self.q_proj(x),   T)
        K   = self._split(self.k_proj(src), src.shape[1])
        V   = self._split(self.v_proj(src), src.shape[1])
        fn  = self._causal_recurrence if (self.causal and mem is None) else self._noncausal
        out = fn(Q, K, V).transpose(1, 2).contiguous().view(B, T, D)
        return self.ln(self.out(self.drop(out)))

# ── Controller ────────────────────────────────────────────────────────────────
class Controller(nn.Module):
    def __init__(self, d):
        super().__init__()
        f = d + 2
        self.wl1 = nn.Linear(f, d // 2); self.wl2 = nn.Linear(d // 2, 1)
        self.fl1 = nn.Linear(f, d)
        self.cl1 = nn.Linear(f, 1)
        self.write_h    = nn.Sequential(self.wl1, nn.GELU(), self.wl2)
        self.forget_h   = nn.Sequential(self.fl1, nn.Sigmoid())
        self.compress_h = nn.Sequential(self.cl1, nn.Sigmoid())
        for l in [self.wl1, self.wl2, self.fl1, self.cl1]:
            nn.init.xavier_uniform_(l.weight)
            nn.init.zeros_(l.bias)
        nn.init.constant_(self.fl1.bias, 2.0)
        nn.init.constant_(self.cl1.bias, 2.0)

    @staticmethod
    def _entropy(x):
        with torch.no_grad():
            B, T, D = x.shape
            q   = F.normalize(x.float(), dim=-1)
            n   = min(T, 64)
            gen = torch.Generator(device=x.device).manual_seed(SEED)
            idx = torch.randperm(T, device=x.device, generator=gen)[:n]
            k   = q[:, idx]
            p   = F.softmax((q @ k.transpose(1, 2)) * (D ** -0.5), dim=-1)
            H   = -(p * (p + 1e-9).log()).sum(-1, keepdim=True)
            return (H / (math.log(n) + 1e-9)).to(x.dtype)

    def forward(self, ao, diag=False, diag_store=None):
        e    = self._entropy(ao)
        norm = ao.norm(dim=-1, keepdim=True) * (ao.shape[-1] ** -0.5)
        feat = torch.cat([ao, e, norm], dim=-1)

        write_logits = self.write_h(feat)
        wg = torch.sigmoid(write_logits).squeeze(-1)
        wg = wg.clamp(min=0.05)

        fg = self.forget_h(feat).mean(1, keepdim=True)
        cg = self.compress_h(feat.mean(1)).squeeze(-1)

        if diag and diag_store is not None:
            diag_store.log("write_gate",    wg.mean().item())
            diag_store.log("forget_gate",   fg.mean().item())
            diag_store.log("compress_gate", cg.mean().item())
            diag_store.log("attn_entropy",  e.mean().item())

        return wg, fg, cg

# ── Semantic Memory ────────────────────────────────────────────────────────────
class SemanticMem(nn.Module):
    def __init__(self, d, n_slots, h):
        super().__init__()
        assert d % h == 0
        self.n, self.h, self.dk = n_slots, h, d // h
        self.sk  = nn.Parameter(torch.randn(n_slots, d) * 0.02)
        self.sv0 = nn.Parameter(torch.randn(n_slots, d) * 0.02)
        self.q_p = nn.Linear(d, d, bias=False)
        self.o_p = nn.Linear(d, d, bias=False)
        self.wg  = nn.Sequential(nn.Linear(d, n_slots), nn.Sigmoid())
        self.wp  = nn.Linear(d, d, bias=False)
        self.ln  = nn.LayerNorm(d)
        for l in [self.q_p, self.o_p, self.wp]:
            nn.init.xavier_uniform_(l.weight)
        for l in self.wg:
            if isinstance(l, nn.Linear):
                nn.init.xavier_uniform_(l.weight)
                nn.init.zeros_(l.bias)

    def forward(self, q, ep_mean):
        B, T, D = q.shape
        H, dk = self.h, self.dk
        sv = self.sv0.unsqueeze(0).expand(B, -1, -1)
        g  = self.wg(ep_mean).unsqueeze(-1)
        sv = (1 - g) * sv + g * self.wp(ep_mean).unsqueeze(1).expand(-1, self.n, -1)
        Q  = self.q_p(q).view(B, T, H, dk).transpose(1, 2)
        K  = self.sk.unsqueeze(0).expand(B, -1, -1).view(B, self.n, H, dk).transpose(1, 2)
        V  = sv.view(B, self.n, H, dk).transpose(1, 2)
        w  = F.softmax((Q @ K.transpose(-2, -1)) * (dk ** -0.5), dim=-1)
        return self.ln(self.o_p((w @ V).transpose(1, 2).contiguous().view(B, T, D)))

# ── Working Memory ─────────────────────────────────────────────────────────────
class WorkingMem(nn.Module):
    def __init__(self, d, n_slots, h, drop=0.1):
        super().__init__()
        self.n, self.d  = n_slots, d
        self.slot_keys  = nn.Parameter(torch.randn(n_slots, d) * 0.02)
        self.write_proj = nn.Linear(d, d, bias=False)
        self.erase_proj = nn.Linear(d, d, bias=False)
        self.read_attn  = LinAttn(d, h, drop, causal=False)
        self.ln         = nn.LayerNorm(d)
        nn.init.xavier_uniform_(self.write_proj.weight)
        nn.init.xavier_uniform_(self.erase_proj.weight)

    def init_state(self, B, dev):
        return torch.zeros(B, self.n, self.d, device=dev)

    def write(self, tokens, state, importance, forget_gate):
        B, T, D = tokens.shape
        wv  = self.write_proj(tokens)
        ev  = torch.sigmoid(self.erase_proj(tokens))
        sk  = self.slot_keys.unsqueeze(0).expand(B, -1, -1)
        with _fp32_ctx():
            addr  = F.softmax(
                torch.bmm(tokens.float(), sk.float().transpose(1, 2)) / math.sqrt(D),
                dim=-1)
            aw    = addr * importance.float().unsqueeze(-1)
            s32   = state.float() * forget_gate.float()
            agg_w = torch.bmm(aw.transpose(1, 2), wv.float())
            agg_e = torch.bmm(aw.transpose(1, 2), ev.float()).clamp(0, 1)
            s32   = s32 * (1 - agg_e) + agg_w
        return s32.to(tokens.dtype)

    def read(self, query, state):
        return self.ln(self.read_attn(query, mem=state))

# ── Episodic Memory ─────────────────────────────────────────────────────────────
class EpisodicMem(nn.Module):
    def __init__(self, d, n_slots, h, drop=0.1):
        super().__init__()
        self.n_short = max(1, n_slots // 2)
        self.n_long  = max(1, n_slots - n_slots // 2)
        self.d       = d
        self.read_attn = LinAttn(d, h, drop, causal=False)
        self.ln        = nn.LayerNorm(d)

    def init_state(self, B, dev):
        return (torch.zeros(B, self.n_short, self.d, device=dev),
                torch.zeros(B, self.n_long,  self.d, device=dev))

    def update(self, short, long, summaries, cg):
        cg    = cg.view(-1, 1, 1).clamp(0, 1).to(short.dtype)
        short = cg * torch.cat([short[:, 1:], summaries[:, :1]], dim=1) + (1 - cg) * short
        ms    = summaries.mean(1, keepdim=True)
        long  = (cg ** 2) * torch.cat([long[:, 1:], ms], dim=1) + (1 - cg ** 2) * long
        return short, long

    def read(self, query, short, long):
        return self.ln(self.read_attn(query, mem=torch.cat([short, long], dim=1)))

# ── Hierarchical Compressor ────────────────────────────────────────────────────
class HComp(nn.Module):
    def __init__(self, d, k, s, t=1.0):
        super().__init__()
        self.k, self.t = k, t
        self.centroids = nn.Parameter(torch.randn(k, d) * 0.02)
        self.sq        = nn.Parameter(torch.randn(1, s, d) * 0.02)
        _n_heads       = max(1, min(4, d // 16))
        self.ap        = nn.MultiheadAttention(d, num_heads=_n_heads,
                                               batch_first=True, dropout=0.)
        self.pj        = nn.Linear(d, d)
        self.ln        = nn.LayerNorm(d)
        nn.init.xavier_uniform_(self.pj.weight)
        nn.init.zeros_(self.pj.bias)

    def forward(self, x):
        dtype = x.dtype; B, T, D = x.shape
        with _fp32_ctx():
            x32 = x.float()
            C32 = self.centroids.float().unsqueeze(0).expand(B, -1, -1)
            d   = (x32.pow(2).sum(-1, True) + C32.pow(2).sum(-1).unsqueeze(1)
                   - 2 * torch.bmm(x32, C32.transpose(1, 2))).clamp(0)
            a   = F.softmax(-d / self.t, dim=-1).to(dtype)
        mass = a.sum(1, keepdim=True).transpose(1, 2).clamp(1e-6)
        cl   = torch.bmm(a.transpose(1, 2), x) / mass
        sm, _ = self.ap(self.sq.expand(B, -1, -1), cl, cl)
        return cl, self.ln(self.pj(sm))

# ── Fusion ─────────────────────────────────────────────────────────────────────
class Fusion(nn.Module):
    def __init__(self, d, drop=0.1):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(d * 3, d), nn.GELU(), nn.Dropout(drop))
        self.ln = nn.LayerNorm(d)
        nn.init.xavier_uniform_(self.fc[0].weight)
        nn.init.zeros_(self.fc[0].bias)

    def forward(self, w, e, s):
        return self.ln(self.fc(torch.cat([w, e, s], dim=-1)))

# ── HMCT Block ─────────────────────────────────────────────────────────────────
class HMCTBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        D, H = cfg.d_model, cfg.n_heads
        self.sa   = LinAttn(D, H, cfg.dropout, causal=cfg.causal)
        self.ctl  = Controller(D)
        self.cmp  = HComp(D, cfg.n_clusters, cfg.n_summaries, cfg.cluster_temp)
        self.work = WorkingMem(D, cfg.working_slots, H, cfg.dropout)
        self.epis = EpisodicMem(D, cfg.episodic_slots, H, cfg.dropout)
        self.sem  = SemanticMem(D, cfg.semantic_slots, H)
        self.fus  = Fusion(D, cfg.dropout)
        self.op   = nn.Sequential(nn.Linear(D * 2, D), nn.GELU(), nn.Dropout(cfg.dropout))
        self.ln   = nn.LayerNorm(D)
        nn.init.xavier_uniform_(self.op[0].weight)
        nn.init.zeros_(self.op[0].bias)

    def _init_state(self, B, dev):
        return {"wb":   self.work.init_state(B, dev),
                "ep_s": self.epis.init_state(B, dev)[0],
                "ep_l": self.epis.init_state(B, dev)[1]}

    def forward(self, x, state=None, diag=False, diag_store=None):
        B, T, D = x.shape
        if state is None: state = self._init_state(B, x.device)

        wb   = state["wb"].detach()
        ep_s = state["ep_s"].detach()
        ep_l = state["ep_l"].detach()

        if diag and diag_store is not None:
            diag_store.log("work_util", (wb.abs().sum(-1) > 1e-3).float().mean().item())
            diag_store.log("epis_util", (ep_s.abs().sum(-1) > 1e-3).float().mean().item())

        ao         = self.sa(x)
        wg, fg, cg = self.ctl(ao, diag=diag, diag_store=diag_store)

        mem_usage  = wg.mean()
        mem_loss   = 0.01 * torch.relu(0.2 - mem_usage)

        wb           = self.work.write(ao, wb, wg, fg)
        w_ctx        = self.work.read(ao, wb)
        _, summaries = self.cmp(ao)
        ep_s, ep_l   = self.epis.update(ep_s, ep_l, summaries, cg)
        e_ctx        = self.epis.read(ao, ep_s, ep_l)
        s_ctx        = self.sem(ao, ep_s.mean(1))

        assert w_ctx.shape == x.shape, f"w_ctx {w_ctx.shape} != x {x.shape}"
        assert e_ctx.shape == x.shape
        assert s_ctx.shape == x.shape

        mc = self.fus(w_ctx, e_ctx, s_ctx)

        assert not mc.isnan().any(), \
            f"NaN in fused memory output. Check attention/compressor numerical stability."

        x = x + 0.5 * mc
        x = x + 0.1 * (x * torch.tanh(mc))

        out = self.ln(x + self.op(torch.cat([ao, mc], dim=-1)))

        assert not out.isnan().any(), "NaN in HMCTBlock output after op+ln"

        return out, {"wb": wb, "ep_s": ep_s, "ep_l": ep_l}, mem_loss, wg

# ── Full HMCT Model ────────────────────────────────────────────────────────────
class HMCT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        D = cfg.d_model
        self.emb  = nn.Embedding(cfg.vocab_size, D)
        self.pos  = nn.Embedding(cfg.max_seq_len, D)
        self.drop = nn.Dropout(cfg.dropout)
        self.blks = nn.ModuleList([HMCTBlock(cfg) for _ in range(cfg.n_layers)])
        self.ffs  = nn.ModuleList([nn.Sequential(
            nn.LayerNorm(D), nn.Linear(D, cfg.ff_dim), nn.GELU(),
            nn.Linear(cfg.ff_dim, D), nn.Dropout(cfg.dropout)
        ) for _ in range(cfg.n_layers)])
        self.ln   = nn.LayerNorm(D)
        self.head = nn.Linear(D, cfg.vocab_size)
        self.diag = Diag()

        nn.init.normal_(self.emb.weight, std=0.02)
        nn.init.normal_(self.pos.weight, std=0.02)
        nn.init.xavier_uniform_(self.head.weight)
        nn.init.zeros_(self.head.bias)
        for ff in self.ffs:
            for m in ff:
                if isinstance(m, nn.Linear):
                    nn.init.xavier_uniform_(m.weight)
                    if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, x, diag=False, return_mem_loss=False):
        B, T = x.shape
        assert T <= self.pos.num_embeddings, \
            f"seq_len {T} > max_seq_len {self.pos.num_embeddings}"
        h = self.drop(self.emb(x) + self.pos(
            torch.arange(T, device=x.device).unsqueeze(0)))
        states         = [None] * len(self.blks)
        total_mem_loss = torch.tensor(0., device=x.device)
        last_wg        = None
        for i, (blk, ff) in enumerate(zip(self.blks, self.ffs)):
            h, states[i], ml, wg = blk(h, states[i], diag=(diag and i == 0),
                                        diag_store=self.diag if diag else None)
            total_mem_loss = total_mem_loss + ml
            if i == 0: last_wg = wg
            h = h + ff(h)
        out = self.head(self.ln(h))
        if return_mem_loss:
            return out, total_mem_loss, last_wg
        return out

# ── Utility ────────────────────────────────────────────────────────────────────
def nparams(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

# ── Smoke test ─────────────────────────────────────────────────────────────────
_h = HMCT(CFG)
_x = torch.randint(0, CFG.vocab_size, (2, 32))
with torch.no_grad():
    _out, _ml, _wg = _h(_x, return_mem_loss=True)
assert _out.shape == (2, 32, CFG.vocab_size), f"Shape error: {_out.shape}"
assert not _out.isnan().any(), "NaN in smoke test output"
assert _wg.min() >= 0.05, f"Write gate below floor: {_wg.min():.4f}"
print(f"✓ HMCT smoke test passed")
print(f"  output={_out.shape}, wg_min={_wg.min():.3f}, mem_loss={_ml.item():.4f}")
del _h, _x, _out, _ml, _wg

✓ HMCT smoke test passed
  output=torch.Size([2, 32, 64]), wg_min=0.050, mem_loss=0.0000


In [20]:
# =========================
# CELL 5 — Baseline Transformer
# =========================

class Baseline(nn.Module):
    """
    Standard causal Transformer — 1 layer, d_model=64.
    Matches HMCT fairness constraints (same d_model, n_layers, ff_dim).
    """
    def __init__(self, cfg):
        super().__init__()
        D, H, FF = cfg.d_model, cfg.n_heads, cfg.ff_dim
        self.emb    = nn.Embedding(cfg.vocab_size, D)
        self.pos    = nn.Embedding(cfg.max_seq_len, D)
        self.drop   = nn.Dropout(cfg.dropout)
        self.layers = nn.ModuleList([nn.ModuleDict({
            "attn": nn.MultiheadAttention(D, H, dropout=cfg.dropout, batch_first=True),
            "ff"  : nn.Sequential(nn.Linear(D, FF), nn.GELU(),
                                  nn.Linear(FF, D), nn.Dropout(cfg.dropout)),
            "n1"  : nn.LayerNorm(D),
            "n2"  : nn.LayerNorm(D),
        }) for _ in range(cfg.n_layers)])
        self.ln   = nn.LayerNorm(D)
        self.head = nn.Linear(D, cfg.vocab_size)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.02)

    def forward(self, x, **kw):
        B, T = x.shape
        # Causal mask
        mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
        h = self.drop(self.emb(x) + self.pos(
            torch.arange(T, device=x.device).unsqueeze(0)))
        for l in self.layers:
            n    = l["n1"](h)
            a, _ = l["attn"](n, n, n, attn_mask=mask)
            h    = h + a
            h    = h + l["ff"](l["n2"](h))
        return self.head(self.ln(h))

# ── Parameter fairness report ─────────────────────────────────────────────────
_h = HMCT(CFG); _b = Baseline(CFG)
_np_h = nparams(_h); _np_b = nparams(_b)
_diff_pct = abs(_np_h - _np_b) / max(_np_b, 1) * 100
print(f"✓ Parameter counts (d_model=64, n_layers=1, working_slots=8):")
print(f"  HMCT     : {_np_h:,}")
print(f"  Baseline : {_np_b:,}")
print(f"  Difference: {_diff_pct:.1f}%")
if _diff_pct > 10:
    print(f"  NOTE: {_diff_pct:.1f}% > 10% target — HMCT has auxiliary memory modules.")
    print(f"  This is inherent to the HMCT architecture; per spec we note but do NOT assert.")
del _h, _b

✓ Parameter counts (d_model=64, n_layers=1, working_slots=8):
  HMCT     : 2,276,720
  Baseline : 623,424
  Difference: 265.2%
  NOTE: 265.2% > 10% target — HMCT has auxiliary memory modules.
  This is inherent to the HMCT architecture; per spec we note but do NOT assert.


In [21]:
# =========================
# CELL 6 — Training Engine
# =========================

class CosineLR:
    def __init__(self, opt, warmup, total, min_lr=1e-6):
        self.opt  = opt; self.ws = warmup; self.ts = total
        self.min  = min_lr; self.n = 0
        self.base = [g["lr"] for g in opt.param_groups]

    def step(self):
        self.n += 1; t = self.n
        for g, lr0 in zip(self.opt.param_groups, self.base):
            if t < self.ws:
                g["lr"] = lr0 * t / max(self.ws, 1)
            else:
                p = (t - self.ws) / max(self.ts - self.ws, 1)
                g["lr"] = self.min + 0.5 * (lr0 - self.min) * (1 + math.cos(math.pi * p))


class InfiniteLoader:
    def __init__(self, loader):
        self.loader = loader
        self._iter  = iter(loader)

    def __next__(self):
        try:
            return next(self._iter)
        except StopIteration:
            self._iter = iter(self.loader)   # only reset when truly exhausted
            return next(self._iter)


def _do_step(model, xb, yb, opt, sch, scaler, is_hmct, cfg):
    """
    Execute one forward + backward + optimizer step.
    Returns (ce_loss_val, acc_val, wg) or raises on NaN.
    """
    xb = xb[:, :cfg.train_seq_len].to(DEVICE, non_blocking=True)
    yb = yb[:, :cfg.train_seq_len].to(DEVICE, non_blocking=True)

    with _amp_ctx():
        if is_hmct:
            logits, mem_loss, wg = model(xb, diag=True, return_mem_loss=True)
            mask    = (yb != 0).view(-1)
            ce_loss = F.cross_entropy(
                logits.view(-1, logits.size(-1))[mask], yb.view(-1)[mask])
            loss = ce_loss + mem_loss
        else:
            logits  = model(xb)
            mask    = (yb != 0).view(-1)
            ce_loss = F.cross_entropy(
                logits.view(-1, logits.size(-1))[mask], yb.view(-1)[mask])
            loss    = ce_loss
            wg      = None

    # Check BEFORE backward — surface the issue rather than hiding it
    if torch.isnan(loss) or torch.isinf(loss):
        return None, None, None   # signal: skip this step

    opt.zero_grad(set_to_none=True)

    if scaler:
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        # Check for bad gradients (inf/nan injected by overflow)
        grad_ok = True
        for p in model.parameters():
            if p.grad is not None:
                if torch.isnan(p.grad).any() or torch.isinf(p.grad).any():
                    grad_ok = False
                    break
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        if grad_ok:
            scaler.step(opt)    # skip opt.step if grads are bad
        scaler.update()
    else:
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        opt.step()

    sch.step()
    acc = scpd_acc(logits.detach(), yb, SCPDDataset.N_MARKERS)
    return ce_loss.item(), acc, wg


def train_epoch(model, inf_loader, opt, sch, scaler, cfg, epoch, is_hmct):
    """
    Train for exactly steps_per_epoch steps using InfiniteLoader.
    """
    model.train()
    if is_hmct: model.diag.reset()

    tl = ta = n = nan_skips = bad_grad_skips = 0
    write_gates = []; work_utils = []
    forget_gates = []; compress_gates = []

    if IS_CUDA:
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

    t0 = time.time()

    for step in range(cfg.steps_per_epoch):
        xb, yb = next(inf_loader)

        try:
            ce_val, acc_val, wg = _do_step(
                model, xb, yb, opt, sch, scaler, is_hmct, cfg)
        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                print(f"  [OOM] step={step}, skipping"); torch.cuda.empty_cache()
                opt.zero_grad(set_to_none=True)
                continue
            raise

        if ce_val is None:
            # NaN loss — count it, log it, but do NOT crash
            nan_skips += 1
            continue

        tl += ce_val; ta += acc_val; n += 1

        if is_hmct and wg is not None:
            wu = (wg > 0.1).float().mean().item()
            write_gates.append(wg.mean().item())
            work_utils.append(wu)
            if is_hmct:
                forget_gates.append(model.diag.mean("forget_gate"))
                compress_gates.append(model.diag.mean("compress_gate"))

    if IS_CUDA: torch.cuda.synchronize()

    elapsed  = time.time() - t0
    gpu_mem  = torch.cuda.max_memory_allocated() / 1e6 if IS_CUDA else 0.
    tps      = (cfg.batch_size * cfg.train_seq_len * max(n, 1)) / max(elapsed, 1e-6)

    # DELAYED VALIDATION (mandatory spec)
    wu_mean = float(np.mean(work_utils)) if work_utils else 0.
    if is_hmct and epoch > 1 and wu_mean < 0.01:
        print(f"  WARNING epoch={epoch}: Memory under-utilized (work_util={wu_mean:.4f})")

    if nan_skips > 0:
        print(f"  [warn] {nan_skips}/{cfg.steps_per_epoch} steps had NaN loss "
              f"(GradScaler will reduce scale; should recover)")

    return {
        "train_loss":         tl / max(n, 1),
        "train_acc":          ta / max(n, 1),
        "time_sec":           elapsed,
        "gpu_mem_mb":         gpu_mem,
        "tokens_per_sec":     tps,
        "write_gate_mean":    float(np.mean(write_gates))   if write_gates   else 0.,
        "forget_gate_mean":   float(np.mean(forget_gates))  if forget_gates  else 0.,
        "compress_gate_mean": float(np.mean(compress_gates))if compress_gates else 0.,
        "work_util":          wu_mean,
        "nan_skips":          nan_skips,
    }


def val_epoch(model, loader, is_hmct, val_seq_len=None):
    model.eval()
    tl = ta = n = 0
    with torch.no_grad():
        for xb, yb in loader:
            if val_seq_len is not None:
                xb = xb[:, :val_seq_len]
                yb = yb[:, :val_seq_len]
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)
            # Guard: if sequence is longer than max_seq_len, skip
            if xb.shape[1] > CFG.max_seq_len:
                continue
            if is_hmct:
                logits, _, _ = model(xb, return_mem_loss=True)
            else:
                logits = model(xb)
            mask = (yb != 0).view(-1)
            if mask.sum() == 0: continue
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1))[mask], yb.view(-1)[mask])
            tl += loss.item()
            ta += scpd_acc(logits, yb, SCPDDataset.N_MARKERS)
            n  += 1
    return tl / max(n, 1), ta / max(n, 1)

print("✓ Training engine defined (InfiniteLoader + NaN-transparent pipeline)")
print(f"  steps_per_epoch={CFG.steps_per_epoch}, train_seq_len={CFG.train_seq_len}")
print(f"  n_train={CFG.n_train} → {CFG.n_train // CFG.batch_size} steps/full_epoch "
      f"→ {CFG.steps_per_epoch / (CFG.n_train // CFG.batch_size) * 100:.0f}% coverage per epoch")

✓ Training engine defined (InfiniteLoader + NaN-transparent pipeline)
  steps_per_epoch=100, train_seq_len=128
  n_train=8000 → 250 steps/full_epoch → 40% coverage per epoch


In [22]:
# =========================
# CELL 7 — Logging
# =========================

LOG_FILE = "logs.json"
ALL_LOGS: List[dict] = []

def save_logs():
    with open(LOG_FILE, "w") as f:
        json.dump(ALL_LOGS, f, indent=2)

def log_epoch(model_name, seq_len, epoch, train_metrics, val_acc, val_loss, is_hmct=False):
    entry = {
        "model":        model_name,
        "seq_len":      seq_len,
        "epoch":        epoch,
        "train_loss":   round(train_metrics["train_loss"], 6),
        "val_loss":     round(val_loss, 6),
        "val_accuracy": round(val_acc, 6),
        "time_sec":     round(train_metrics["time_sec"], 3),
        "gpu_mem_mb":   round(train_metrics["gpu_mem_mb"], 2),
        "tokens_per_sec": round(train_metrics["tokens_per_sec"], 1),
        "nan_skips":    train_metrics.get("nan_skips", 0),
    }
    if is_hmct:
        entry["write_gate_mean"]    = round(train_metrics["write_gate_mean"], 6)
        entry["forget_gate_mean"]   = round(train_metrics["forget_gate_mean"], 6)
        entry["compress_gate_mean"] = round(train_metrics["compress_gate_mean"], 6)
        entry["work_util"]          = round(train_metrics["work_util"], 6)
    ALL_LOGS.append(entry)
    save_logs()   # Save after EVERY epoch (mandatory spec)
    return entry

def print_log(entry, is_hmct=False):
    nan_w = f" ⚠NaN={entry['nan_skips']}" if entry.get("nan_skips", 0) > 0 else ""
    print(f"  [{entry['model']}] ep={entry['epoch']}  "
          f"loss={entry['train_loss']:.4f}  val_acc={entry['val_accuracy']:.3f}  "
          f"t={entry['time_sec']:.1f}s  mem={entry['gpu_mem_mb']:.0f}MB  "
          f"tps={entry['tokens_per_sec']:.0f}{nan_w}")
    if is_hmct:
        print(f"    wg={entry['write_gate_mean']:.3f}  "
              f"fg={entry['forget_gate_mean']:.3f}  "
              f"cg={entry['compress_gate_mean']:.3f}  "
              f"util={entry['work_util']:.3f}")

print(f"✓ Logging system ready → saves to {LOG_FILE} after every epoch")

✓ Logging system ready → saves to logs.json after every epoch


In [23]:
# =========================
# CELL 8 — Experiments
# =========================

# ─── A. Main Training: HMCT vs Baseline ──────────────────────────────────────
print("=" * 70)
print("  EXPERIMENT A — Main Training (HMCT vs Baseline, nl=10)")
print(f"  epochs={CFG.epochs}, steps/epoch={CFG.steps_per_epoch}, "
      f"batch={CFG.batch_size}, train_seq_len={CFG.train_seq_len}")
print(f"  n_train={CFG.n_train}, InfiniteLoader active — all samples reachable")
print("=" * 70)

NL_MAIN = 10   # seq_len = 1+4+10+1+4 = 20, well within 128

tr_loader, vl_loader, te_loader = make_loaders(NL_MAIN, CFG)
inf_tr = InfiniteLoader(tr_loader)   # SINGLE iterator shared across all epochs

# ── Train HMCT ────────────────────────────────────────────────────────────────
hmct_model = HMCT(CFG).to(DEVICE)
try:
    opt_h = torch.optim.AdamW(hmct_model.parameters(), lr=CFG.lr,
                               weight_decay=CFG.weight_decay, fused=IS_CUDA)
except TypeError:
    opt_h = torch.optim.AdamW(hmct_model.parameters(), lr=CFG.lr,
                               weight_decay=CFG.weight_decay)
sch_h    = CosineLR(opt_h, CFG.warmup_steps, CFG.steps_per_epoch * CFG.epochs)
scaler_h = _make_scaler()
best_hmct_acc = 0.; best_hmct_sd = None

print("\n── Training HMCT ──")
for ep in range(1, CFG.epochs + 1):
    # InfiniteLoader is NOT reset here — avoids the data siloing bug
    train_m = train_epoch(hmct_model, inf_tr, opt_h, sch_h, scaler_h, CFG, ep, is_hmct=True)
    vl_loss, vl_acc = val_epoch(hmct_model, vl_loader, is_hmct=True)
    entry = log_epoch("HMCT", CFG.train_seq_len, ep, train_m, vl_acc, vl_loss, is_hmct=True)
    print_log(entry, is_hmct=True)
    if vl_acc > best_hmct_acc:
        best_hmct_acc = vl_acc
        best_hmct_sd  = copy.deepcopy(hmct_model.state_dict())

if best_hmct_sd:
    hmct_model.load_state_dict(best_hmct_sd)
    print(f"  ↑ Best HMCT restored, val_acc={best_hmct_acc:.4f}")

# ── Train Baseline ─────────────────────────────────────────────────────────────
baseline_model = Baseline(CFG).to(DEVICE)
tr_loader_b, vl_loader_b, te_loader_b = make_loaders(NL_MAIN, CFG)
inf_tr_b = InfiniteLoader(tr_loader_b)   # separate InfiniteLoader for baseline

try:
    opt_b = torch.optim.AdamW(baseline_model.parameters(), lr=CFG.lr,
                               weight_decay=CFG.weight_decay, fused=IS_CUDA)
except TypeError:
    opt_b = torch.optim.AdamW(baseline_model.parameters(), lr=CFG.lr,
                               weight_decay=CFG.weight_decay)
sch_b    = CosineLR(opt_b, CFG.warmup_steps, CFG.steps_per_epoch * CFG.epochs)
scaler_b = _make_scaler()
best_base_acc = 0.; best_base_sd = None

print("\n── Training Baseline ──")
for ep in range(1, CFG.epochs + 1):
    train_m = train_epoch(baseline_model, inf_tr_b, opt_b, sch_b, scaler_b,
                          CFG, ep, is_hmct=False)
    vl_loss, vl_acc = val_epoch(baseline_model, vl_loader_b, is_hmct=False)
    entry = log_epoch("Baseline", CFG.train_seq_len, ep, train_m, vl_acc, vl_loss, is_hmct=False)
    print_log(entry, is_hmct=False)
    if vl_acc > best_base_acc:
        best_base_acc = vl_acc
        best_base_sd  = copy.deepcopy(baseline_model.state_dict())

if best_base_sd:
    baseline_model.load_state_dict(best_base_sd)
    print(f"  ↑ Best Baseline restored, val_acc={best_base_acc:.4f}")

print(f"\n✓ A. HMCT={best_hmct_acc:.4f}  Baseline={best_base_acc:.4f}  "
      f"Δ={best_hmct_acc - best_base_acc:+.4f}")


# ─── B. Long-context generalisation ─────────────────────────────────────────
print("\n" + "=" * 70)
print("  EXPERIMENT B — Long-Context Generalisation")
print("  Trained on nl=10 (seq~20). Evaluated on nl=20,40 (seq~30,50).")
print("  FIX: val_epoch no longer truncates to 128 — uses actual seq len.")
print("=" * 70)

# DO NOT reinitialise models — use trained weights
long_ctx_results = {}
for test_nl in [20, 40]:
    actual_len = 1 + SCPDDataset.N_MARKERS + test_nl + 1 + SCPDDataset.N_MARKERS
    _lc_ds_h  = DataLoader(SCPDDataset(400, test_nl, CFG.vocab_size, 99),
                            batch_size=CFG.batch_size, collate_fn=pad_collate)
    _lc_ds_b  = DataLoader(SCPDDataset(400, test_nl, CFG.vocab_size, 99),
                            batch_size=CFG.batch_size, collate_fn=pad_collate)
    # Pass val_seq_len=None so sequences are NOT truncated
    _, ta_h = val_epoch(hmct_model,    _lc_ds_h, is_hmct=True,  val_seq_len=None)
    _, ta_b = val_epoch(baseline_model, _lc_ds_b, is_hmct=False, val_seq_len=None)
    long_ctx_results[test_nl] = {"hmct": round(ta_h, 4), "base": round(ta_b, 4)}
    log_epoch("HMCT_longctx", actual_len, 0,
              {"train_loss": 0, "train_acc": 0, "time_sec": 0, "gpu_mem_mb": 0,
               "tokens_per_sec": 0, "write_gate_mean": 0, "forget_gate_mean": 0,
               "compress_gate_mean": 0, "work_util": 0},
              ta_h, 0, is_hmct=True)
    log_epoch("Baseline_longctx", actual_len, 0,
              {"train_loss": 0, "train_acc": 0, "time_sec": 0, "gpu_mem_mb": 0,
               "tokens_per_sec": 0},
              ta_b, 0)
    print(f"  nl={test_nl} (seq_len={actual_len}):  "
          f"HMCT={ta_h:.3f}  Baseline={ta_b:.3f}  Δ={ta_h - ta_b:+.3f}")

print("✓ B. Long-context done")


# ─── C. Ablation — disable working memory ───────────────────────────────────
print("\n" + "=" * 70)
print("  EXPERIMENT C — Ablation (Working Memory Disabled)")
print("  FIX: Previously zeroed s['wb'] AFTER the block output — meaningless")
print("  for n_layers=1. Now patches WorkingMem.read to return zeros.")
print("=" * 70)

# DO NOT reinitialise — patch trained HMCT at the module level
class _ZeroReadWorkingMem(WorkingMem):
    """Ablation patch: read always returns zeros. write() still runs normally."""
    def read(self, query, state):
        return torch.zeros_like(query)

def _make_ablated_hmct(trained_model):
    # Swap WorkingMem class on the trained model's block
    abl = copy.deepcopy(trained_model)
    for blk in abl.blks:
        # Replace .read method on the instance
        orig_read = blk.work.read
        def _zero_read(query, state, _orig=orig_read):
            return torch.zeros_like(query)
        blk.work.read = _zero_read
    return abl

ablated_model = _make_ablated_hmct(hmct_model)
ablated_model.eval()
_, abl_acc = val_epoch(ablated_model, vl_loader, is_hmct=True)
print(f"  Full HMCT  val_acc={best_hmct_acc:.4f}")
print(f"  No-Working val_acc={abl_acc:.4f}  Δ={best_hmct_acc - abl_acc:+.4f}")
ablation_delta = best_hmct_acc - abl_acc
log_epoch("HMCT_ablated_nowork", CFG.train_seq_len, 0,
          {"train_loss": 0, "train_acc": 0, "time_sec": 0,
           "gpu_mem_mb": 0, "tokens_per_sec": 0, "write_gate_mean": 0,
           "forget_gate_mean": 0, "compress_gate_mean": 0, "work_util": 0},
          abl_acc, 0, is_hmct=True)
print(f"✓ C. Working memory contributed Δ={ablation_delta:+.4f}")


# ─── D. Stability ────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("  EXPERIMENT D — Stability Analysis (NaN tracking, loss curves)")
print("=" * 70)

_hmct_logs = [e for e in ALL_LOGS if e["model"] == "HMCT"]
_base_logs  = [e for e in ALL_LOGS if e["model"] == "Baseline"]
_hl = [e["train_loss"] for e in _hmct_logs]
_bl = [e["train_loss"] for e in _base_logs]
_hn = sum(1 for v in _hl if not math.isfinite(v))
_bn = sum(1 for v in _bl if not math.isfinite(v))
total_nan_h = sum(e.get("nan_skips", 0) for e in _hmct_logs)
total_nan_b = sum(e.get("nan_skips", 0) for e in _base_logs)
print(f"  HMCT  NaN epochs: {_hn}/{len(_hl)}  NaN steps total: {total_nan_h}")
print(f"  Base  NaN epochs: {_bn}/{len(_bl)}  NaN steps total: {total_nan_b}")
if _hl: print(f"  HMCT  loss: {_hl[0]:.4f} → {_hl[-1]:.4f}")
if _bl: print(f"  Base  loss: {_bl[0]:.4f} → {_bl[-1]:.4f}")
print("✓ D. Stability done")


# ─── E. Internal Analysis ─────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("  EXPERIMENT E — Internal Analysis (Gates, Gradient Norms per Module)")
print("=" * 70)

hmct_model.train()
_xb_e, _yb_e = next(iter(tr_loader))
_xb_e = _xb_e[:, :CFG.train_seq_len].to(DEVICE)
_yb_e = _yb_e[:, :CFG.train_seq_len].to(DEVICE)

opt_h.zero_grad()
with _amp_ctx():
    _logits_e, _ml_e, _wg_e = hmct_model(_xb_e, diag=True, return_mem_loss=True)
    _mask_e = (_yb_e != 0).view(-1)
    _loss_e = (F.cross_entropy(_logits_e.view(-1, _logits_e.size(-1))[_mask_e],
                               _yb_e.view(-1)[_mask_e]) + _ml_e)
_loss_e.backward()

grad_norms = {}
for name, param in hmct_model.named_parameters():
    if param.grad is not None:
        grad_norms[name] = param.grad.norm().item()

top_norms = sorted(grad_norms.items(), key=lambda kv: kv[1], reverse=True)[:5]
print("  Top 5 gradient norms:")
for nm, gn in top_norms:
    print(f"    {nm:<55} {gn:.5f}")

zero_grad = [nm for nm, p in hmct_model.named_parameters()
             if p.requires_grad and p.grad is None]
if zero_grad:
    print(f"  ⚠ {len(zero_grad)} params with None grad: {zero_grad[:3]}")
else:
    print(f"  ✓ All {sum(p.requires_grad for p in hmct_model.parameters())} "
          f"trainable params have gradients")

_wg_trend = [e["write_gate_mean"] for e in _hmct_logs if "write_gate_mean" in e]
_wu_trend  = [e["work_util"]       for e in _hmct_logs if "work_util"       in e]
if _wg_trend:
    print(f"  Gate trends: write_gate {_wg_trend[0]:.3f}→{_wg_trend[-1]:.3f}  "
          f"work_util {_wu_trend[0]:.3f}→{_wu_trend[-1]:.3f}")

opt_h.zero_grad()
print("✓ E. Internal analysis done")


# ─── F. Memory Influence Check (MANDATORY) ────────────────────────────────────
print("\n" + "=" * 70)
print("  EXPERIMENT F — Memory Influence Check (MANDATORY)")
print("  FIX: Now uses _make_ablated_hmct which patches WorkingMem.read,")
print("  ensuring memory is actually zeroed BEFORE it reaches the output.")
print("=" * 70)

hmct_model.eval()
_xb_f = torch.randint(0, CFG.vocab_size, (4, CFG.train_seq_len), device=DEVICE)

with torch.no_grad():
    out_with_mem,    _, _wg_f = hmct_model(_xb_f, return_mem_loss=True)
    _ablated_inf = _make_ablated_hmct(hmct_model)
    out_without_mem, _, _     = _ablated_inf(_xb_f, return_mem_loss=True)

diff = (out_with_mem - out_without_mem).abs().mean().item()
print(f"  output diff (with memory vs without): {diff:.6f}")
if diff < 1e-6:
    print("  ✗ CRITICAL: Memory has zero influence — something is wrong")
elif diff < 1e-3:
    print(f"  ⚠ Memory influence small ({diff:.6f}) — model may not be fully trained")
else:
    print(f"  ✓ Memory actively influences output (diff={diff:.4f})")


# ─── A2. Scaling Experiment ────────────────────────────────────────────────────
print("\n── Scaling Experiment ──")
seq_lens_scale = [128, 256, 512]
scale_results  = {}

# ONE fixed random batch tensor, resized by slicing ONLY — no regeneration
_fixed_batch = torch.randint(0, CFG.vocab_size,
                              (CFG.batch_size, max(seq_lens_scale)), device=DEVICE)
hmct_model.eval()

for sl in seq_lens_scale:
    _xb_s = _fixed_batch[:, :sl]   # slice — same tensor, no new data
    if IS_CUDA:
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
    t0 = time.time()
    with torch.no_grad():
        _out_s, _, _ = hmct_model(_xb_s, return_mem_loss=True)
    if IS_CUDA: torch.cuda.synchronize()
    elapsed_s = time.time() - t0
    mem_s     = torch.cuda.max_memory_allocated() / 1e6 if IS_CUDA else 0.
    tps_s     = (CFG.batch_size * sl) / max(elapsed_s, 1e-6)
    scale_results[sl] = {"time": elapsed_s, "mem_mb": mem_s, "tps": tps_s}
    print(f"  seq_len={sl:4d}:  t={elapsed_s:.3f}s  mem={mem_s:.1f}MB  tps={tps_s:.0f}")
    ALL_LOGS.append({"model": "HMCT_scale", "seq_len": sl, "epoch": 0,
                     "train_loss": 0, "val_loss": 0, "val_accuracy": 0,
                     "time_sec": round(elapsed_s, 4), "gpu_mem_mb": round(mem_s, 2),
                     "tokens_per_sec": round(tps_s, 1), "nan_skips": 0})
    save_logs()

print("✓ F. Memory influence & scaling done")

  EXPERIMENT A — Main Training (HMCT vs Baseline, nl=10)
  epochs=10, steps/epoch=100, batch=32, train_seq_len=128
  n_train=8000, InfiniteLoader active — all samples reachable

── Training HMCT ──
  [HMCT] ep=1  loss=4.1488  val_acc=0.000  t=21.7s  mem=211MB  tps=18834
    wg=0.409  fg=0.836  cg=0.940  util=0.975
  [HMCT] ep=2  loss=3.3757  val_acc=0.000  t=21.8s  mem=211MB  tps=18752
    wg=0.270  fg=0.839  cg=0.932  util=0.607
  [HMCT] ep=3  loss=2.6593  val_acc=0.003  t=20.9s  mem=211MB  tps=19585
    wg=0.149  fg=0.840  cg=0.903  util=0.201
  [HMCT] ep=4  loss=2.2518  val_acc=0.014  t=20.8s  mem=220MB  tps=19653
    wg=0.151  fg=0.839  cg=0.840  util=0.230
  [HMCT] ep=5  loss=1.8130  val_acc=0.147  t=20.5s  mem=220MB  tps=19936
    wg=0.173  fg=0.840  cg=0.739  util=0.316
  [HMCT] ep=6  loss=1.0421  val_acc=0.911  t=20.1s  mem=220MB  tps=20402
    wg=0.219  fg=0.841  cg=0.608  util=0.420
  [HMCT] ep=7  loss=0.3035  val_acc=0.999  t=20.0s  mem=220MB  tps=20464
    wg=0.221  fg=0.84

In [24]:
# =========================
# CELL 9 — Plotting
# =========================

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

os.makedirs("plots", exist_ok=True)

# STRICT RULE: read from logs.json only — do NOT recompute metrics
with open(LOG_FILE) as f:
    logs = json.load(f)

_h_logs = [e for e in logs if e["model"] == "HMCT"]
_b_logs = [e for e in logs if e["model"] == "Baseline"]

# ── Plot 1: Loss & Accuracy curves ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle("HMCT v8 vs Baseline — Training Dynamics", fontsize=13)

if _h_logs:
    eps = [e["epoch"] for e in _h_logs]
    axes[0].plot(eps, [e["train_loss"]  for e in _h_logs], "b-o",  label="HMCT Train")
    axes[0].plot(eps, [e["val_loss"]    for e in _h_logs], "b--s", label="HMCT Val")
if _b_logs:
    eps = [e["epoch"] for e in _b_logs]
    axes[0].plot(eps, [e["train_loss"]  for e in _b_logs], "r-o",  label="Base Train")
    axes[0].plot(eps, [e["val_loss"]    for e in _b_logs], "r--s", label="Base Val")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].set_title("Loss"); axes[0].legend(); axes[0].grid(True, alpha=0.3)

if _h_logs:
    eps = [e["epoch"] for e in _h_logs]
    axes[1].plot(eps, [e["val_accuracy"] for e in _h_logs], "b-o", label="HMCT Val Acc")
if _b_logs:
    eps = [e["epoch"] for e in _b_logs]
    axes[1].plot(eps, [e["val_accuracy"] for e in _b_logs], "r-o", label="Base Val Acc")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy")
axes[1].set_title("Validation Accuracy"); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("plots/loss_accuracy.png", dpi=100, bbox_inches="tight")
plt.close()
print("✓ plots/loss_accuracy.png")

# ── Plot 2: Memory gate trends ────────────────────────────────────────────────
_hg = [e for e in logs if e["model"] == "HMCT" and "write_gate_mean" in e]
if _hg:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    fig.suptitle("HMCT v8 — Memory Gate Trends", fontsize=13)
    ep_g = [e["epoch"] for e in _hg]
    axes[0].plot(ep_g, [e["write_gate_mean"]    for e in _hg], "g-o")
    axes[0].axhline(0.05, color="r", ls="--", label="floor=0.05"); axes[0].legend()
    axes[0].set_title("Write Gate"); axes[0].set_xlabel("Epoch"); axes[0].grid(True, alpha=0.3)
    axes[1].plot(ep_g, [e["forget_gate_mean"]   for e in _hg], "m-o")
    axes[1].set_title("Forget Gate"); axes[1].set_xlabel("Epoch"); axes[1].grid(True, alpha=0.3)
    axes[2].plot(ep_g, [e["compress_gate_mean"] for e in _hg], "c-o")
    axes[2].set_title("Compress Gate"); axes[2].set_xlabel("Epoch"); axes[2].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("plots/gate_trends.png", dpi=100, bbox_inches="tight")
    plt.close()
    print("✓ plots/gate_trends.png")

# ── Plot 3: Work utilisation ──────────────────────────────────────────────────
if _hg:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(ep_g, [e["work_util"] for e in _hg], "b-o", label="work_util")
    ax.axhline(0.10, color="r",      ls="--", label="threshold=0.10")
    ax.axhline(0.01, color="orange", ls=":",  label="warning=0.01")
    ax.set_title("HMCT Working Memory Utilisation")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Fraction(gates>0.1)")
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("plots/work_utilisation.png", dpi=100, bbox_inches="tight")
    plt.close()
    print("✓ plots/work_utilisation.png")

# ── Plot 4: GPU memory & throughput ──────────────────────────────────────────
if _h_logs and _b_logs:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle("GPU Memory & Throughput per Epoch", fontsize=13)
    axes[0].plot([e["epoch"] for e in _h_logs], [e["gpu_mem_mb"]     for e in _h_logs], "b-o", label="HMCT")
    axes[0].plot([e["epoch"] for e in _b_logs], [e["gpu_mem_mb"]     for e in _b_logs], "r-o", label="Baseline")
    axes[0].set_title("GPU Memory (MB)"); axes[0].legend(); axes[0].grid(True, alpha=0.3)
    axes[1].plot([e["epoch"] for e in _h_logs], [e["tokens_per_sec"] for e in _h_logs], "b-o", label="HMCT")
    axes[1].plot([e["epoch"] for e in _b_logs], [e["tokens_per_sec"] for e in _b_logs], "r-o", label="Baseline")
    axes[1].set_title("Tokens / sec"); axes[1].legend(); axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("plots/gpu_throughput.png", dpi=100, bbox_inches="tight")
    plt.close()
    print("✓ plots/gpu_throughput.png")

# ── Plot 5: Scaling ────────────────────────────────────────────────────────────
_sc = [e for e in logs if e["model"] == "HMCT_scale"]
if _sc:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle("HMCT v8 — Scaling (fixed batch, sliced)", fontsize=13)
    sls = [e["seq_len"] for e in _sc]
    axes[0].plot(sls, [e["time_sec"]    for e in _sc], "b-o"); axes[0].set_title("Time (s)")
    axes[0].set_xlabel("Seq Len"); axes[0].grid(True, alpha=0.3)
    axes[1].plot(sls, [e["gpu_mem_mb"] for e in _sc], "r-o"); axes[1].set_title("GPU Mem (MB)")
    axes[1].set_xlabel("Seq Len"); axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("plots/scaling.png", dpi=100, bbox_inches="tight")
    plt.close()
    print("✓ plots/scaling.png")

# ── Plot 6: NaN step counts ────────────────────────────────────────────────────
_nan_h = [e.get("nan_skips", 0) for e in _h_logs]
_nan_b = [e.get("nan_skips", 0) for e in _b_logs]
if any(_nan_h) or any(_nan_b):
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar([e["epoch"] for e in _h_logs], _nan_h, alpha=0.6, label="HMCT", color="blue")
    ax.bar([e["epoch"] for e in _b_logs], _nan_b, alpha=0.6, label="Baseline", color="red")
    ax.set_title("NaN Steps per Epoch (should be 0 with init_scale=128)")
    ax.set_xlabel("Epoch"); ax.set_ylabel("NaN steps"); ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("plots/nan_counts.png", dpi=100, bbox_inches="tight")
    plt.close()
    print("✓ plots/nan_counts.png (NaN steps detected and logged)")

print("\n✓ All plots generated")

✓ plots/loss_accuracy.png
✓ plots/gate_trends.png
✓ plots/work_utilisation.png
✓ plots/gpu_throughput.png
✓ plots/scaling.png

✓ All plots generated


In [25]:
# =========================
# CELL 10 — Results Table
# =========================

print("\n" + "=" * 70)
print("  RESULTS TABLE")
print("=" * 70)

with open(LOG_FILE) as f:
    logs_t = json.load(f)

_np_h = nparams(hmct_model); _np_b = nparams(baseline_model)

print(f"\n  Parameters:  HMCT={_np_h:,}  Baseline={_np_b:,}  "
      f"Δ={abs(_np_h-_np_b)/_np_b*100:.1f}%")
print()

print(f"  {'Model':<24} {'SeqLen':>7} {'Time(s)':>9} {'Mem(MB)':>9} {'ValAcc':>9} {'NaN/ep':>8}")
print("  " + "-" * 68)

for mn in ["HMCT", "Baseline"]:
    for e in [e for e in logs_t if e["model"] == mn]:
        print(f"  {mn:<24} {e['seq_len']:>7} {e['time_sec']:>9.2f} "
              f"{e['gpu_mem_mb']:>9.1f} {e['val_accuracy']:>9.4f} "
              f"{e.get('nan_skips', 0):>8}")

print()
print(f"  {'Model':<24} {'SeqLen':>7} {'ValAcc':>9} {'Note'}")
print("  " + "-" * 60)
for mn, note in [("HMCT_longctx", "long-ctx HMCT"),
                  ("Baseline_longctx", "long-ctx Base"),
                  ("HMCT_ablated_nowork", "ablation -work")]:
    for e in [e for e in logs_t if e["model"] == mn]:
        print(f"  {mn:<24} {e['seq_len']:>7} {e['val_accuracy']:>9.4f}  {note}")

print()
print(f"  Scaling (HMCT, fixed batch sliced):")
print(f"  {'SeqLen':>10} {'Time(s)':>10} {'Mem(MB)':>10} {'TPS':>10}")
for e in [e for e in logs_t if e["model"] == "HMCT_scale"]:
    print(f"  {e['seq_len']:>10} {e['time_sec']:>10.4f} {e['gpu_mem_mb']:>10.1f} "
          f"{e['tokens_per_sec']:>10.0f}")

print()
print(f"  Memory influence diff: {diff:.6f}")
print(f"  Ablation Δ (full - no_work): {ablation_delta:+.4f}")


  RESULTS TABLE

  Parameters:  HMCT=2,276,720  Baseline=623,424  Δ=265.2%

  Model                     SeqLen   Time(s)   Mem(MB)    ValAcc   NaN/ep
  --------------------------------------------------------------------
  HMCT                         128     21.75     211.3    0.0000        0
  HMCT                         128     21.84     211.3    0.0000        0
  HMCT                         128     20.91     211.3    0.0031        0
  HMCT                         128     20.84     220.4    0.0138        0
  HMCT                         128     20.55     220.4    0.1475        0
  HMCT                         128     20.08     220.4    0.9113        0
  HMCT                         128     20.02     220.4    0.9994        0
  HMCT                         128     20.90     220.4    1.0000        0
  HMCT                         128     21.03     220.4    1.0000        0
  HMCT                         128     20.85     220.4    1.0000        0
  Baseline                     128    

In [26]:
# =========================
# CELL 11 — Export
# =========================

import shutil

os.makedirs("outputs", exist_ok=True)

torch.save(hmct_model.state_dict(),     "outputs/hmct.pt")
torch.save(baseline_model.state_dict(), "outputs/baseline.pt")
print("✓ Models saved: hmct.pt, baseline.pt")

shutil.copy(LOG_FILE, "outputs/logs.json")
print(f"✓ Logs saved: {len(ALL_LOGS)} entries")

for fname in os.listdir("plots"):
    shutil.copy(os.path.join("plots", fname), os.path.join("outputs", fname))
print(f"✓ Plots saved: {os.listdir('plots')}")

zip_path = "results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir("outputs"):
        zf.write(os.path.join("outputs", fname), fname)
print(f"✓ Zipped → {zip_path}")

try:
    from google.colab import files
    files.download(zip_path)
    print("✓ Auto-download triggered")
except ImportError:
    print(f"  (Not Colab — download {zip_path} manually)")

✓ Models saved: hmct.pt, baseline.pt
✓ Logs saved: 28 entries
✓ Plots saved: ['gpu_throughput.png', 'work_utilisation.png', 'gate_trends.png', 'scaling.png', 'loss_accuracy.png']
✓ Zipped → results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Auto-download triggered


In [27]:
# =========================
# CELL 12 — Final Validation
# =========================

print("\n" + "=" * 70)
print("  FINAL VALIDATION")
print("=" * 70)

errors = []; warns = []

# 1. No NaNs in model outputs
hmct_model.eval(); baseline_model.eval()
with torch.no_grad():
    _xv  = torch.randint(0, CFG.vocab_size, (2, 32), device=DEVICE)
    _ov, _ml_v, _wg_v = hmct_model(_xv, return_mem_loss=True)
    _ob  = baseline_model(_xv)
    if _ov.isnan().any():  errors.append("NaN in HMCT output")
    else:                  print("  ✓ No NaNs in HMCT output")
    if _ob.isnan().any():  errors.append("NaN in Baseline output")
    else:                  print("  ✓ No NaNs in Baseline output")

# 2. Logs exist and are non-empty
with open(LOG_FILE) as f: _fl = json.load(f)
if len(_fl) > 0:
    print(f"  ✓ logs.json: {len(_fl)} entries, {os.path.getsize(LOG_FILE)} bytes")
else:
    errors.append("logs.json is empty")

# 3. Plots exist
_pf = os.listdir("plots") if os.path.isdir("plots") else []
if _pf: print(f"  ✓ Plots: {_pf}")
else:   errors.append("No plots found")

# 4. Models saved
for mf in ["outputs/hmct.pt", "outputs/baseline.pt"]:
    if os.path.exists(mf): print(f"  ✓ {mf} ({os.path.getsize(mf)//1024} KB)")
    else:                  errors.append(f"{mf} missing")

# 5. Memory output ≠ 0 (uses the FIXED ablation method)
with torch.no_grad():
    _xm      = torch.randint(0, CFG.vocab_size, (2, 64), device=DEVICE)
    _om, _, _ = hmct_model(_xm, return_mem_loss=True)
    _oa, _, _ = _make_ablated_hmct(hmct_model)(_xm, return_mem_loss=True)
    _mem_diff = (_om - _oa).abs().mean().item()
if _mem_diff > 1e-6:
    print(f"  ✓ memory_output ≠ 0 (diff={_mem_diff:.6f})")
else:
    errors.append(f"memory_output ≈ 0 ({_mem_diff:.2e}) — memory has no influence")

# 6. work_util > 0.01
_wu_list = [e["work_util"] for e in _fl if "work_util" in e]
if _wu_list:
    _wu_last = _wu_list[-1]
    if _wu_last > 0.01: print(f"  ✓ work_util={_wu_last:.4f} > 0.01")
    else:               warns.append(f"work_util={_wu_last:.4f} ≤ 0.01")
else:
    warns.append("No work_util entries in logs")

# 7. Write gate floor
_wg_min = _wg_v.min().item()
if _wg_min >= 0.05: print(f"  ✓ write_gate min={_wg_min:.4f} ≥ 0.05 (floor enforced)")
else:               errors.append(f"write_gate_min={_wg_min:.4f} < 0.05")

# 8. NaN skip count
_total_nan = sum(e.get("nan_skips", 0) for e in _fl if e["model"] in ("HMCT", "Baseline"))
if _total_nan == 0:
    print(f"  ✓ Zero NaN steps across all epochs (init_scale=128 working)")
else:
    warns.append(f"{_total_nan} NaN steps occurred — GradScaler recovered")

# 9. Data coverage check
_steps_seen = CFG.steps_per_epoch * CFG.epochs
_samples_seen = _steps_seen * CFG.batch_size
print(f"  ✓ Data coverage: {_steps_seen} steps × {CFG.batch_size} = "
      f"{_samples_seen:,} sample visits / {CFG.n_train:,} unique samples "
      f"({_samples_seen / CFG.n_train * 100:.0f}% coverage with InfiniteLoader)")

# ── Summary ───────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
if errors:
    print(f"  ✗ FAILED — {len(errors)} error(s):")
    for e in errors: print(f"    • {e}")
else:
    print("  ✓ ALL CHECKS PASSED")

if warns:
    print(f"  ⚠ {len(warns)} warning(s):")
    for w in warns: print(f"    • {w}")

_np_h = nparams(hmct_model); _np_b = nparams(baseline_model)

# FIX: Safely format the work_util value
_wu_formatted = f"{_wu_list[-1]:.4f}" if _wu_list else "N/A"

print(f"""
  FINAL SUMMARY
  ─────────────────────────────────────────────────────
  HMCT  best val_acc : {best_hmct_acc:.4f}
  Base  best val_acc : {best_base_acc:.4f}
  Δ accuracy         : {best_hmct_acc - best_base_acc:+.4f}
  Memory influence   : {_mem_diff:.6f}
  Ablation Δ (work)  : {ablation_delta:+.4f}
  Params HMCT        : {_np_h:,}
  Params Baseline    : {_np_b:,}
  Param Δ            : {abs(_np_h-_np_b)/_np_b*100:.1f}%
  work_util (last)   : {_wu_formatted}
  NaN steps total    : {_total_nan}
  Plots saved        : {len(_pf)}
  Log entries        : {len(_fl)}
""")
print("✓ HMCT v8 complete.")


  FINAL VALIDATION
  ✓ No NaNs in HMCT output
  ✓ No NaNs in Baseline output
  ✓ logs.json: 28 entries, 8391 bytes
  ✓ Plots: ['gpu_throughput.png', 'work_utilisation.png', 'gate_trends.png', 'scaling.png', 'loss_accuracy.png']
  ✓ outputs/hmct.pt (8974 KB)
  ✓ outputs/baseline.pt (2453 KB)
  ✓ memory_output ≠ 0 (diff=0.635841)
  ✓ write_gate min=0.0500 ≥ 0.05 (floor enforced)
  ✓ Zero NaN steps across all epochs (init_scale=128 working)
  ✓ Data coverage: 1000 steps × 32 = 32,000 sample visits / 8,000 unique samples (400% coverage with InfiniteLoader)

  ✓ ALL CHECKS PASSED
  ⚠ 1 warning(s):
    • work_util=0.0000 ≤ 0.01

  FINAL SUMMARY
  ─────────────────────────────────────────────────────
  HMCT  best val_acc : 1.0000
  Base  best val_acc : 1.0000
  Δ accuracy         : +0.0000
  Memory influence   : 0.635841
  Ablation Δ (work)  : +0.0000
  Params HMCT        : 2,276,720
  Params Baseline    : 623,424
  Param Δ            : 265.2%
  work_util (last)   : 0.0000
  NaN steps total 

In [31]:
# =========================
# CELL 13 — Display Architecture (FAST + FIXED SIZE)
# =========================

# 1. Install visualization libraries (only runs once, safe to keep)
!pip install -q torchview graphviz

import torch
from torchview import draw_graph
from IPython.display import Image, display

print("Generating architecture diagram...")

# 2. Dummy input (Batch=1, SeqLen=32)
dummy_input = torch.randint(0, CFG.vocab_size, (1, 32), device=DEVICE)

# 3. Generate graph (NO saving → fast)
model_graph = draw_graph(
    hmct_model,
    input_data=dummy_input,
    graph_name="HMCT_Architecture",
    depth=2,              # ↓ reduce for speed + clarity (use 1 if still slow)
    save_graph=False
)

# 4. Render as PNG in-memory and display at fixed width (pixels)
png_bytes = model_graph.visual_graph.pipe(format='png')

display(Image(png_bytes, width=900))  # change width (e.g., 800 / 1000)

Generating architecture diagram...
